# Testing out the redaction capabilities


In [1]:
import os
import sys
import pandas as pd

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if root_path not in sys.path:
    sys.path.insert(0, root_path)
for line in sys.path: 
    print(line)

c:\Users\danie\Code\aiticketgrouper
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64\python313.zip
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64\DLLs
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64\Lib
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64
c:\Users\danie\Code\aiticketgrouper\.venv

c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages
c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages\win32
c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages\win32\lib
c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages\Pythonwin


In [2]:
csv_path = "../../external_files/incident_ticket_data.csv" # go up two levels to get external files

df = pd.read_csv(csv_path)

if "Description" not in df.columns:
    raise ValueError("CSV must contain 'Description' column")

df = df[:1000]

df.head()

,Number,Description,Opened by,Company,ITSM Department,Created,Urgency,Impact,Priority,Assignment group,...,User input,Comments and Work notes,Manday Effort (hrs),Ticket Type,AMS Domain,AMS System Type,AMS Category Type,AMS Service Type,AMS Business Related,AMS IT Related
0,INC0010171,GRPT not working as expected. ZMMM_PO_REV is n...,Indah Humairah Sulaiman,PIDSAP,PIDSAP,18/3/24 9:07,2 - Medium,3 - Low,4 - Low,PISCAP L2 SD BRS,...,NaN,2025-04-11 13:26:58 - BALAKUMAR GANESAN (Addit...,3.0,Issue,IS,S4HANA,Non-Genesis,Business-Related,BZ-B12-Master Data (Wrong Maintenance),NaN
1,INC0010181,eTR-S1-24000073\r\nExchange Rate did not auto ...,Indah Humairah Sulaiman,PIDSAP,PIDSAP,18/3/24 9:51,2 - Medium,3 - Low,4 - Low,PISCAP L2 Workflow (SN),...,NaN,2024-04-05 02:49:36 - Reeman Mathur (Additiona...,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN
2,INC0010188,There is no GRPT maintenance for Sold-To: 3901...,Indah Humairah Sulaiman,PIDSAP,PIDSAP,18/3/24 10:19,3 - Low,3 - Low,4 - Low,PISCAP L2 SD BRS,...,NaN,2024-05-13 12:57:15 - BALAKUMAR GANESAN (Addit...,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN
3,INC0010189,Interface\t fpl\r\nSubsidiary\t...,Chenxing Cao,PA,PISCAP,18/3/24 10:24,3 - Low,3 - Low,4 - Low,PISCAP L2 Mulesoft/SOA,...,NaN,2024-03-18 10:30:07 - Chenxing Cao (Work notes...,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN
4,INC0010192,"retrieve new SAP password, thank you.",SOOK FONG NG,PM,PM,18/3/24 10:33,1 - High,3 - Low,3 - Moderate,PISCAP L2 SAP BASIS,...,NaN,NaN,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.shape

(1000, 30)

!python -m spacy download en_core_web_sm

In [4]:
import re
from clustering_service.config import Settings
from clustering_service.ticket_processing.models.ticket_batch import TicketProcess, TicketBatchProcessData
from clustering_service.data_cleaning.cleaning import CleaningService
from clustering_service.data_cleaning.redaction import RedactionService

redact = RedactionService()
settings = Settings()
cleaner = CleaningService(settings)

print(f"Cleaning Pattern: {cleaner.pattern}")

ticket_list: list[TicketProcess] = []
redacted_data_list = []
cleaned_data_list = []
# df = df[['Number', 'Description', 'Comments and Work notes', 'Created']]
df = df[['Number', 'Description', 'Created']]


Cleaning Pattern: re.compile('\\b(asap|cr|vs|pc|com|po|p\\.o\\.|p\\ o|p/o|sfdc)\\b', re.IGNORECASE)


In [5]:
test_str = """PO# 100039583  Delivery Instruction is not pulling correctly based on PIR .  
PO pull Lorry 05 instead of Air Frt . 
Technically if incoterm do not have alternative record, it should pull from PIR. 
PIR maintenance Air Frt 01 
Incoterm - No record 
Enclosed pls find Incoterm table ,"""
test_str = cleaner.clean_text(test_str)
print(f"Cleaned String: {test_str}")
test_str = redact.redact_text(test_str)
print(f"Redacted String: {test_str}")

Cleaned String: <purchase_order_id> delivery instruction is not pulling correctly based on purchase information record . purchase order pull lorry 05 instead of air frt . technically if incoterm do not have alternative record, it should pull from purchase information record. purchase information record maintenance air frt 01 incoterm - no record enclosed find incoterm table ,
Redacted String: <purchase_order_id> delivery instruction is not pulling correctly based on purchase information record . purchase order pull lorry 05 instead of air frt . technically if incoterm do not have alternative record, it should pull from purchase information record. purchase information record maintenance air frt 01 incoterm - no record enclosed find incoterm table ,


In [6]:
test_str="""Please include these names in eWorkplace (see attached file for your reference):
IIEE Ilocos Norte Chapter
1 Rigor Gabur - President
2 Wilfred Paz - VP Internal
3 Kervin Borja - VP External
4 Salustiano Morales - VP Technical
5 Melwyn Pascua - Secretary
6 Rommel Agcaoili - Treasurer

IIEE Baguio Chapter
1 Mario Esteban - President
2 Mario Calatan - VP Internal
3 Aileen Arevalo - VP Technical
4 Gelia Elegado - Secretary
5 Erwin Posadas - Auditor
6 Vincent Madino - BOD"""
test_str = cleaner.clean_text(test_str)
test_str = redact.redact_text(test_str)
print(f"Redacted String: {test_str}")

Redacted String: include these names in eworkplace (see attached file for your reference): iiee <LOCATION> chapter 1 rigor gabur - president 2 <PERSON> internal 3 <PERSON> external 4 salustiano <PERSON> technical 5 melwyn pascua - secretary 6 <PERSON> - treasurer iiee <LOCATION> chapter 1 mario esteban - president 2 <PERSON> internal 3 <PERSON> technical 4 gelia elegado - secretary 5 <PERSON> 6 <PERSON> - bod


In [7]:
test_str = """GRPT not working as expected. ZMMM_PO_REV is not generating correct dates as per maintained in GRPT table. 
E.g. P/O# 100024066
Vendor Ship mode is 03. 
As per GRPT route days are 12 days and GR days is 0 days. Based on ex-fty date which 22 Nov 2024, correct ETA and ETD date is 4 Dec 2024 but system showing 27 Nov 2024 instead. Pls kindly advise."""
test_str = cleaner.clean_text(test_str)
test_str = redact.redact_text(test_str)
print(f"Redacted String: {test_str}")

Redacted String: grpt not working as expected. zmmm_po_rev is not generating correct dates as per maintained in grpt table. e.g. <purchase_order_id> vendor ship mode is 03. as per grpt route days are 12 days and gr days is 0 days. based on ex-fty date which 22 nov 2024, correct eta and etd date is 4 dec 2024 but system showing 27 nov 2024 instead. advise.


In [8]:
test_str = """Unable to edit S/O 110024937. eSO posting failed due to time out. Pls see image attached. 
HTTP POST on resource 'https://vhpmkmbpci.hec.privatecloudedition.com:44300/sap/opu/odata/sap/YMSD_SAPEWORK_API_SRV/inESOSet/' failed: Timeout exceeded.","""
test_str = cleaner.clean_text(test_str)
test_str = redact.redact_text(test_str)
print(f"Redacted String: {test_str}")

Redacted String: unable to edit <sales_order_id>. eso posting failed due to time out. see image attached. http post on resource 'https://vhpmkmbpci.hec.privatecloudedition.computer:44300/sap/opu/odata/sap/ymsd_sapework_api_srv/inesoset/' failed: timeout exceeded.",


In [9]:

for index, row in df.iterrows():
    input_str = row["Description"]
    cleaned_data = cleaner.clean_text(input_str)
    ticket_list.append(TicketProcess(
        id=int(str.replace(row["Number"], "INC", "")),
        description=row["Description"],
        created_at=row["Created"],
        embed_str=cleaned_data
    ))
    cleaned_data_list.append(cleaned_data)

    
df['cleaned_text'] = cleaned_data_list
df.head()



,Number,Description,Created,cleaned_text
0,INC0010171,GRPT not working as expected. ZMMM_PO_REV is n...,18/3/24 9:07,grpt not working as expected. zmmm_po_rev is n...
1,INC0010181,eTR-S1-24000073\r\nExchange Rate did not auto ...,18/3/24 9:51,etr-s1-24000073 exchange rate did not auto pop...
2,INC0010188,There is no GRPT maintenance for Sold-To: 3901...,18/3/24 10:19,there is no grpt maintenance for sold-to: 3901...
3,INC0010189,Interface\t fpl\r\nSubsidiary\t...,18/3/24 10:24,interface fpl subsidiary pidsap api name pana-...
4,INC0010192,"retrieve new SAP password, thank you.",18/3/24 10:33,"retrieve new sap password, ."


In [10]:
for index, row in df.iterrows():
    combined_str = row["cleaned_text"] if row["cleaned_text"] is not None else row["Description"]
    redact_data = redact.redact_text(combined_str)
    redacted_data_list.append(redact_data)

df['redacted_text'] = redacted_data_list
df.head()

,Number,Description,Created,cleaned_text,redacted_text
0,INC0010171,GRPT not working as expected. ZMMM_PO_REV is n...,18/3/24 9:07,grpt not working as expected. zmmm_po_rev is n...,grpt not working as expected. zmmm_po_rev is n...
1,INC0010181,eTR-S1-24000073\r\nExchange Rate did not auto ...,18/3/24 9:51,etr-s1-24000073 exchange rate did not auto pop...,etr-s1-24000073 exchange rate did not auto pop...
2,INC0010188,There is no GRPT maintenance for Sold-To: 3901...,18/3/24 10:19,there is no grpt maintenance for sold-to: 3901...,there is no grpt maintenance for sold-to: 3901...
3,INC0010189,Interface\t fpl\r\nSubsidiary\t...,18/3/24 10:24,interface fpl subsidiary pidsap api name pana-...,interface fpl subsidiary pidsap api name pana-...
4,INC0010192,"retrieve new SAP password, thank you.",18/3/24 10:33,"retrieve new sap password, .","retrieve new sap password, ."


Redaction Script

In [12]:
output_path = "../../external_files/incident_ticket_data_redacted.csv"
df.to_csv(output_path, index=False)
print(f"DataFrame saved to {output_path}")

DataFrame saved to ../../external_files/incident_ticket_data_redacted.csv
